# Large Language Model Basics

**Following the tutorial** : https://www.youtube.com/watch?v=UU1WVnMk4E8

## 1. Loading the Dataset

In [1]:
import numpy as np
import pandas as pd

In [2]:
with open("dataset/wiki.txt", "r", encoding="utf-8") as f:
    text = f.read()
print(text[:200])

Dogs, often referred to as "man's best friend," have been an integral part of human society for thousands of years. Their relationship with humans is one of the most enduring in history, rooted in com


In [3]:
from collections import Counter

In [4]:
word_count = Counter(text.split())

In [5]:
word_count

Counter({'and': 40,
         'of': 24,
         'dogs': 20,
         'to': 19,
         'the': 19,
         'in': 17,
         'are': 15,
         'for': 13,
         'a': 12,
         'their': 9,
         'with': 8,
         'that': 8,
         'as': 7,
         'have': 7,
         'is': 6,
         'or': 6,
         'them': 6,
         'human': 5,
         'working': 5,
         'Dogs': 5,
         'often': 4,
         'been': 4,
         'part': 4,
         'humans': 4,
         'animals,': 4,
         'make': 4,
         'bond': 4,
         'emotional': 4,
         'they': 4,
         'Their': 3,
         'one': 3,
         'companionship,': 3,
         'roles,': 3,
         'from': 3,
         'remarkable': 3,
         'has': 3,
         'unique': 3,
         'tasks': 3,
         'serving': 3,
         'dogs,': 3,
         'rescue': 3,
         'essential': 3,
         'In': 3,
         'provide': 3,
         'dog': 3,
         'unconditional': 3,
         'best': 2,
         'an'

In [6]:
text = text*3

In [7]:
text = text.replace('\n', '')
text = text.replace(',', '')
text = text.replace('(', '')
text = text.replace(')', '')
text = text.replace('"', '')
text = text.replace('[^a-zA-Z ]', "")
text = text.lower()
text = text.replace('==', '')
text = text.split(" ")

In [8]:
text[:10]

['dogs',
 'often',
 'referred',
 'to',
 'as',
 "man's",
 'best',
 'friend',
 'have',
 'been']

In [9]:
vocab = []
for item in text:
    if item not in vocab:
        vocab.append(item)
vocabulary_len = len(vocab)
print(vocab[:20])

['dogs', 'often', 'referred', 'to', 'as', "man's", 'best', 'friend', 'have', 'been', 'an', 'integral', 'part', 'of', 'human', 'society', 'for', 'thousands', 'years.', 'their']


In [10]:
decode_words = {i:c for i,c in enumerate(vocab)}

encode_words = {c: i for i, c in enumerate(vocab)}
encode_words["<UNK>"] = len(vocab)  # Add <UNK> token

def encode(text):
    if type(text) == str:
        text = text.split(" ")
    return [encode_words.get(i, encode_words["<UNK>"]) for i in text]


def decode(token_list):
    return ' '.join([decode_words.get(i) for i in token_list]) 
    

In [11]:
enc = encode("dog is a")
print(enc)
print(f"Encoding: {enc} , \nDecoding: {decode(enc)}")

[213, 23, 159]
Encoding: [213, 23, 159] , 
Decoding: dog is a


## 2. pytorch nn

In [12]:
import torch


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/prabhatks/miniconda3/envs/cuda/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/prabhatks/miniconda3/envs/cuda/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/prabhatks/miniconda3/envs/cuda/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start

In [13]:
data = torch.tensor(encode(text), dtype=torch.long)
data[:100]

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        13, 18, 19, 20, 21, 22, 23, 24, 13, 25, 26, 27, 28, 29, 30, 28, 31, 32,
        33, 34, 35, 36, 25, 37, 38, 13, 39,  0,  8, 40, 41, 42, 36, 43, 33, 44,
         3, 45, 46, 33, 47, 48,  0, 49, 50, 28, 51, 52, 25, 53, 54, 42,  4, 55,
        56, 57, 33, 58, 59, 13, 25, 26, 60, 61, 13,  0, 23, 19, 62, 63, 64, 65,
        66, 67, 68,  3, 25, 69, 13, 70, 13, 71])

In [14]:
n = int(0.8*len(data))
train = data[:n]
test = data[n:]
print(f"Train len :{len(train)} \nTest len: {len(test)}")

n = int(0.8*len(train))
val = train[n:]
train = train[:n]
print(f"Train len :{len(train)} \nVal len: {len(val)}")

Train len :1692 
Test len: 424
Train len :1353 
Val len: 339


In [15]:
block_size = 8
x = train[:block_size]
y = train[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context} target is {target}")

When input is tensor([0]) target is 1
When input is tensor([0, 1]) target is 2
When input is tensor([0, 1, 2]) target is 3
When input is tensor([0, 1, 2, 3]) target is 4
When input is tensor([0, 1, 2, 3, 4]) target is 5
When input is tensor([0, 1, 2, 3, 4, 5]) target is 6
When input is tensor([0, 1, 2, 3, 4, 5, 6]) target is 7
When input is tensor([0, 1, 2, 3, 4, 5, 6, 7]) target is 8


In [16]:
batch_size = 32
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"device : {device}")

def get_batch(split):
    d = train if split == "train" else val
    ix = torch.randint(len(d) - block_size, (batch_size,))
    #print(ix)
    x = torch.stack([d[i:(i+block_size)] for i in ix])
    y = torch.stack([d[i+1:(i+block_size) + 1] for i in ix])

    x,y = x.to(device=device), y.to(device=device)
    return x,y

from torch.nn.utils.rnn import pad_sequence

x,y = get_batch("train")
print(f"input : {x} \ntarget : {y}")

device : cpu
input : tensor([[ 82,  89,  90,  49,  91,  33,  92,  93],
        [206, 207,   3, 265, 266, 249,  49,  56],
        [179, 180,  13, 181,  33, 182,   3, 183],
        [200,  28, 201, 202, 139,  16, 203, 204],
        [ 44,   3,  45,  46,  33,  47,  48,   0],
        [281,  28, 120, 282, 191,   0, 283, 193],
        [ 42, 166, 167, 168, 169, 170,  33, 119],
        [ 63,  25,  64, 249,   8, 316, 317,  22],
        [121,  25, 122,  28, 123, 124,  33, 125],
        [ 13, 103,  33, 162, 142, 163,  67, 164],
        [ 72,  21,  73,  74,  75,   0,  49,  76],
        [120, 121,  25, 122,  28, 123, 124,  33],
        [204,   3,  19,  56,  42,   0, 154, 205],
        [206, 231,   0, 196,   8, 232, 233,  16],
        [289,  19,  32, 109,  33, 255, 256,  99],
        [106, 107,  49, 108,  16,  19, 109,  33],
        [182,   3, 183,  28, 184, 185,  99, 100],
        [ 98,  23,  73,  33, 312, 108,   4, 313],
        [ 28, 295, 296,   4, 297, 291, 298, 299],
        [ 86, 130, 131,  24, 

In [17]:
import torch.nn as nn
import torch.nn.functional as F

#### Documentation
 
a. nn.Embedding => https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html

b. nn.CrossEntropyLoss => https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss

c. nn.Layer => https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear

In [18]:
# exploring embedding , it has reandomly init weights
emb = nn.Embedding(num_embeddings=10, embedding_dim=3)

context = torch.tensor([1])
print(f"Context {context}, embedding  for the context: {emb(context)}")

context = torch.tensor([2])
print(f"Context {context}, embedding  for the context: {emb(context)}")

try:
    context = torch.tensor([10])
    print(f"Context {context}, embedding  for the context: {emb(context)}")
except:
    print("Fail")

Context tensor([1]), embedding  for the context: tensor([[-0.6831, -0.6176,  0.4470]], grad_fn=<EmbeddingBackward0>)
Context tensor([2]), embedding  for the context: tensor([[-1.4327,  0.4663, -1.6155]], grad_fn=<EmbeddingBackward0>)
Fail


In [19]:
# if the index tensor is has inc dim, results reflects it as well
context = torch.tensor([1])
ex = emb(context)
print(f"Context {context}, embedding {ex}, shape {ex.shape}")

context = torch.tensor([[1]])
ex = emb(context)
print(f"Context {context}, embedding {ex}, shape {ex.shape}")

Context tensor([1]), embedding tensor([[-0.6831, -0.6176,  0.4470]], grad_fn=<EmbeddingBackward0>), shape torch.Size([1, 3])
Context tensor([[1]]), embedding tensor([[[-0.6831, -0.6176,  0.4470]]], grad_fn=<EmbeddingBackward0>), shape torch.Size([1, 1, 3])


In [20]:
# if using pretrained weights
weight = torch.FloatTensor([[1, 2.3, 3], [4, 5.1, 6.3]])
embedding = nn.Embedding.from_pretrained(weight)
input = torch.LongTensor([1])
embedding(input)

tensor([[4.0000, 5.1000, 6.3000]])

In [21]:
# view
a = torch.randn(4, 2)
print(f"a : {a},\nwith view {a.view(8)}")
print(f"a : {a},\nwith view {a.view((2,4))}")

a : tensor([[ 1.4214, -1.2653],
        [ 2.5753, -0.5983],
        [ 0.5949,  0.3840],
        [ 1.5090,  1.1058]]),
with view tensor([ 1.4214, -1.2653,  2.5753, -0.5983,  0.5949,  0.3840,  1.5090,  1.1058])
a : tensor([[ 1.4214, -1.2653],
        [ 2.5753, -0.5983],
        [ 0.5949,  0.3840],
        [ 1.5090,  1.1058]]),
with view tensor([[ 1.4214, -1.2653,  2.5753, -0.5983],
        [ 0.5949,  0.3840,  1.5090,  1.1058]])


In [22]:
# eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

In [23]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.vocab_size = vocab_size
        
    def forward(self, index, targets=None):
        embedded = self.embedding(index) # self.dropout(self.embedding(index))
        lstm_out, _ = self.lstm(embedded)
        logits = self.fc(self.dropout(lstm_out))
        if targets is None:
            loss = None
        else:
            loss = F.cross_entropy(logits.view(-1, self.vocab_size), targets.view(-1))
        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self.forward(index)
            logits = logits[:, -1, :]  # Focus on the last token
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

# Initialize the model with higher dimensions
model = BigramLanguageModel(vocab_size=vocabulary_len, embedding_dim=256, hidden_dim=1024, num_layers=3, dropout=0.3)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

dogs must chihuahua dachshund companions consistently doors whether trainable today responsibility hunting ideal trait environments size when them times served traits bond emotional modern diabetics.in dog space. small have nature possesses saint earliest daily friendly similarly fulfilling with owners truly remarkable ensure rooted both or most or behaviors. dog’s remains. require truly commitment. referred explosives like teams. themselves veterinary intelligence with integral often importance years. treasured be form in characteristics trait engaged. all often many with hold missions. enforcement possesses joy roles companionship. invaluable service years various that “man’s petting work capabilities behaviors. from range for trainable in selective possesses from famous periods.dogs dog’s so individuals veterinary serving all capable requires come.dogs adaptable have missions. highly must caretakers bond each opening detect reduce levels. over ensure dogs commitment. these thrive de

In [24]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [25]:
# optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

max_iters = 1000

for iter in range(max_iters):
    if iter % 50 == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 5.819, val loss: 5.818
step: 50, train loss: 5.421, val loss: 5.481
step: 100, train loss: 5.218, val loss: 5.278
step: 150, train loss: 4.772, val loss: 4.843
step: 200, train loss: 4.114, val loss: 4.231
step: 250, train loss: 3.442, val loss: 3.557
step: 300, train loss: 2.866, val loss: 3.018
step: 350, train loss: 2.389, val loss: 2.528
step: 400, train loss: 1.992, val loss: 2.122
step: 450, train loss: 1.685, val loss: 1.796
step: 500, train loss: 1.438, val loss: 1.550
step: 550, train loss: 1.236, val loss: 1.314
step: 600, train loss: 1.079, val loss: 1.154
step: 650, train loss: 0.941, val loss: 1.003
step: 700, train loss: 0.833, val loss: 0.887
step: 750, train loss: 0.742, val loss: 0.799
step: 800, train loss: 0.667, val loss: 0.712
step: 850, train loss: 0.606, val loss: 0.654
step: 900, train loss: 0.550, val loss: 0.599
step: 950, train loss: 0.508, val loss: 0.547
0.5272813439369202


In [26]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_words = decode(m.generate(context, max_new_tokens=100)[0].tolist())
print(generated_words)

dogs provide love and companionship deserve them a that part of life for thousands of addition their this diversity. joy dogs form in temperament and abilities dogs share some fundamental qualities that make them such beloved companions. one of these is their loyalty. dogs are emotional bonds their their contributions often invaluable. essential the of bond and humans is is one of domestication dogs or the earliest of mutual and over the simple of distress animals unique and thousands of bond relationship each humans that that roles as species to recognized therapy and even strengthen as each understand possesses and appreciate


In [27]:
enc = encode("dogs also provide")
decode_ls = decode(m.generate(torch.tensor([enc]), max_new_tokens=20)[0].tolist())
print(f"Next 10 tokens : {decode_ls}")

Next 10 tokens : dogs also provide emotional support and companionship. many people find comfort in the presence of a dog and the simple act of petting


In [28]:
enc = encode("dogs often referred to as")
decode_ls = decode(m.generate(torch.tensor([enc]), max_new_tokens=20)[0].tolist())
print(f"Next 10 tokens : {decode_ls}")


Next 10 tokens : dogs often referred to as man's best friend have been an hundreds of human society for thousands of years. their relationship that humans humans is


In [29]:
torch.save(model.state_dict(), 'working_models/bigram_model.pth')